# P8 1차 — READ Transcoder 24개 학습·반환 감사

GPU 런타임을 선택하고 위부터 실행합니다. `v1_4_p8_training_r2_progress.zip`을 **내 드라이브/boolean_interp_v1_4/**에 업로드해 두세요. 기존 P5 원본 `P5_r2`를 입력으로 읽고 새 결과는 `P8_training_r1`에 저장합니다.

세 LM × block 0·3·7·11 × k=4/16, sparse seed 0입니다. 각 TC는 같은 위치의 MLP 입력 u에서 MLP 출력 m을 예측합니다. 대응 SAE와 같은 position draw 순서, 5,000 updates/512 positions, 250 updates마다 전체 validation MSE 선택 규칙을 유지합니다.

**이 노트북은 학습 단계입니다.** 학습 결과를 반환·감사한 뒤 동결한 규칙대로 의미·fidelity·대체·인과 평가를 이어갑니다. 학습만으로 P8을 완료하지 않습니다. Test activation이나 상태 라벨은 학습·checkpoint 선택에 쓰지 않습니다.

중단되면 같은 노트북을 다시 실행합니다. 마지막 checksum이 있는 checkpoint에서 optimizer·RNG·draw cursor를 복구합니다. 다른 환경이면 새 환경 ID와 GPU smoke가 필요합니다. 완료 run과 기존 결과는 보존합니다.

**r2 진행 표시 추가:** 학습 셀에서 전체/현재 run 진행률, 경과시간, 속도와 예상 남은 시간을 표시합니다. r1 학습 코드·계약·출력 경로는 동일하여 기존 checkpoint를 재개할 수 있습니다. ETA는 학습 단계의 추정이며 후속 검산·평가 시간은 제외됩니다.

In [ ]:
from google.colab import files, drive
from pathlib import Path
import hashlib, zipfile, subprocess, sys, json, time, shutil, os
drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/boolean_interp_v1_4')
BUNDLE_IN_DRIVE=BASE/'v1_4_p8_training_r2_progress.zip'
bundle=Path('/content/v1_4_p8_training_r2_progress.zip')
def digest(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()
expected='6701fba8db6d606f860579aef30a0ff9addbcb15292d73537d4f7d7ca61b0155'
if not bundle.exists() or digest(bundle)!=expected:
    assert BUNDLE_IN_DRIVE.exists(), '입력 ZIP을 Drive 경로에 업로드하세요'
    shutil.copyfile(BUNDLE_IN_DRIVE,bundle)
assert digest(bundle)==expected, '입력 ZIP checksum 불일치'
ROOT=Path('/content/MI_P8_training_r1');ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(bundle) as z:
    for item in z.infolist():assert (ROOT/item.filename).resolve().is_relative_to(ROOT.resolve()), '잘못된 ZIP 경로'
    z.extractall(ROOT)
SOURCE=BASE/'P5_r2'
OUTPUT=BASE/'P8_training_r1';OUTPUT.mkdir(parents=True,exist_ok=True)
assert (SOURCE/'contract.json').exists(), 'P5 원본 cache 경로 확인 필요'
print('입력 ZIP checksum 검증 완료', bundle.stat().st_size, 'bytes')


## 1. 의존성·환경 준비
실행 환경은 P6/P7과 같은 버전을 설치합니다. 설치 로그·pip freeze·GPU 정보·환경 ID를 저장합니다. 재시작 안내가 나오면 런타임 재시작 후 첫 셀부터 실행하세요.

In [ ]:
log=subprocess.run([sys.executable,'-m','pip','install','numpy==2.1.3','scipy==1.16.3','pytest==8.4.2','torch==2.11.0','--extra-index-url','https://download.pytorch.org/whl/cu128'],text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
(OUTPUT/f'install_{time.time_ns()}.txt').write_text(log.stdout)
print(log.stdout);log.check_returncode()
os.environ['OMP_NUM_THREADS']='2';os.environ['OPENBLAS_NUM_THREADS']='2'
sys.path.insert(0,str(ROOT))
from interp_v1_4.p8 import verify,prepare,export
verify(ROOT)
overlay=json.loads((ROOT/'p8_progress_overlay.json').read_text())
for name,checksum in overlay['files'].items():
    assert digest(ROOT/name)==checksum, '진행 표시 코드 checksum 불일치'
subprocess.run([sys.executable,'-m','pytest','tests_v1_4/test_p8.py','-q'],cwd=ROOT,check=True)


## 2. 원본 checksum·READ key·입출력 전처리
240개 cache checksum을 확인하고 train 50k/val 10k의 네 층 u/m을 준비합니다. 36개 train scalar 통계가 감사된 P6 값과 정확히 같은지 대조합니다. 대형 P5 입력은 Drive에서 읽으므로 시간이 걸립니다. test는 checksum만 확인하며 학습·선택용 tensor로 만들지 않습니다.

In [ ]:
manifest=prepare(ROOT,SOURCE,OUTPUT)
print('검증 완료:',manifest['statistics_count'],'통계;',len(manifest['tensors']),'train/val tensors')


## 3. 현재 코드의 CUDA smoke와 정확한 재개
네 층의 hook·identity/full/sparse patch, debug dictionary 100 updates, TC k=4/16의 model·optimizer·draw RNG가 중단 없는 실행과 bitwise 일치하는지 확인합니다. 입력 s_u와 출력 s_m은 서로 다른 값으로 검사합니다. Debug 데이터는 본학습에 재사용하지 않습니다.

In [ ]:
SMOKE=OUTPUT/'smoke'/str(time.time_ns())
subprocess.run([sys.executable,'-m','interp_v1_4.p8','smoke','--root',str(ROOT),'--output',str(SMOKE),'--device','cuda'],cwd=ROOT,check=True)


## 4. TC 24 runs 학습·재개
최초 100 updates의 처리량·peak VRAM 측정은 5,000 updates에 포함됩니다. 250 updates마다 저장·validation 평가합니다. 마지막 평가까지 끝난 후 최소 val MSE checkpoint를 선택하며 동률은 이른 update입니다. 재실행하면 완료 run은 확인하고 중단 run은 마지막 완전 저장 경계에서 재개합니다.

In [ ]:
RUNS=None  # 일부 실행 예: ['seed0_l0_tc_k4_s0']; 완료에는 24개 모두 필요
from scripts.p8_progress import run_with_progress
command=[sys.executable,'-m','interp_v1_4.p8','train','--root',str(ROOT),'--output',str(OUTPUT),'--smoke',str(SMOKE/'smoke.json'),'--device','cuda']+(['--runs']+RUNS if RUNS else [])
progress=run_with_progress(command,ROOT,OUTPUT,RUNS)


## 5. 전체 학습 수치 검산
24개가 모두 끝난 뒤 실행합니다. 504개 checkpoint, 480개 validation MSE, optimizer·RNG cursor·unit norm·입출력 통계·최소 MSE 선택을 독립 검산합니다. F.linear·float32 tensor scale·stable TopK의 수치 경로를 유지합니다. 검산은 선택 규칙을 변경하지 않습니다. 중단 상태는 이 셀을 건너뛰고 다운로드하세요.

In [ ]:
subprocess.run([sys.executable,str(ROOT/'scripts/verify_v1_4_p8_training.py'),'--root',str(ROOT),'--output',str(OUTPUT),'--device','cuda'],cwd=ROOT,check=True)


## 6. 결과 다운로드
완료 또는 중단 뒤 실행합니다. 모든 checkpoint와 optimizer/RNG, 입력 manifest·통계·환경·검산·실패 기록을 포함합니다. ZIP과 checksum JSON을 이 작업에 전달하세요. Drive 결과는 보존합니다. 재생성 가능한 u/m tensor와 원본 P5 cache는 ZIP에 중복 포함하지 않습니다. 선택 TC 감사 후 별도 평가 노트북으로 이어갑니다.

In [ ]:
archive=Path(f'/content/v1_4_p8_training_return_{time.time_ns()}.zip')
export(OUTPUT,archive)
print('SHA256:',digest(archive))
files.download(str(archive))
files.download(str(archive.with_suffix('.sha256.json')))
